# Expert reference polygons (*Acacia dealbata*)

**Purpose.** Look at the expert data on their own to decide whether
and how to use them (part 1), then clean them (part 2).

**Data.** Two GeoPackages sent by the supervisor (2026-10), in
`data/raw/expert_reference/` (see `data/README.md`):
- `TNE_17MAY2026_5.gpkg`: Sentinel-2 tile TNE, which covers the
  Coimbra district;
- `notTNE_18MAY2026_6.gpkg`: the rest of continental Portugal.

What we were told: polygons of *A. dealbata*, collected by visual
interpretation and/or field work; use only `classifica = 1`.
Answered by the supervisor (2026-10-28): 2023, from field visits and
manual visual interpretation of OrtoSat 2023; iNaturalist did not
influence the collection.

**Part 1 — exploration (nothing changed):** 1. what is in the files;
2. geometry quality and how they were made; 3. sizes and shapes;
4. where they are (and overlap with iNaturalist); 5. what COS 2023 says is under each polygon.

**Part 2 — cleaning:** 6. keep only `classifica = 1` rows; 7. fix
invalid geometries; 8. overlaps; 9. add attributes; 10. save.

**Decision update (2026-11-04, supervisor):** only `classifica = 1`
polygons are used in this study; `classifica = 0` ("not *A.
dealbata*") polygons are dropped, not used as absences — see
[`thesis/notes.md`](../../thesis/notes.md#usar-os-dados-de-especialistas-com-limpeza).
Part 1 below still explores the full raw data (including
`classifica = 0`), since that exploration is what surfaced the
building-bias concern behind this decision; only part 2 (cleaning)
changed.

## 0. Setup

Loads both files as they are, adds `file` (which file each row came
from), and the district outline from iNaturalist (place 8422). All in
EPSG:3763 (PT-TM06, metres).

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import shape

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
EXPERT = ROOT / "data" / "raw" / "expert_reference"
FILES = {"TNE": "TNE_17MAY2026_5.gpkg", "notTNE": "notTNE_18MAY2026_6.gpkg"}

raw = gpd.GeoDataFrame(pd.concat([gpd.read_file(EXPERT / f).assign(file=k) for k, f in FILES.items()],
                                 ignore_index=True), crs=3763)
place = requests.get("https://api.inaturalist.org/v1/places/8422", timeout=60).json()["results"][0]
district = gpd.GeoSeries([shape(place["geometry_geojson"])], crs=4326).to_crs(3763).iloc[0]
print(f"{len(raw)} rows")

## 1. What is in the files

- Columns: only `classifica` and the geometry. No date, no method
  (field or visual), no author, no species field.
- `classifica`: stored as text in one file and as a number in the
  other, so it is converted to a number here. `NaN` means empty.

In [ ]:
raw["classifica"] = pd.to_numeric(raw["classifica"], errors="coerce")
print("columns:", [c for c in raw.columns if c != "geometry"])
print("geometry types:", raw.geom_type.groupby(raw["file"]).value_counts().to_dict())
pd.crosstab(raw["file"], raw["classifica"].fillna(-1).astype(int).replace(-1, "empty"), margins=True)

## 2. Geometry quality

Checks that a GIS would complain about or that would bias counts:
- **invalid:** self-intersecting or broken polygons;
- **empty:** no geometry;
- **multipart:** one row made of several separate pieces (counted
  as one polygon unless split);
- **exact duplicates:** the same geometry twice;
- **overlapping positives:** two `classifica = 1` polygons sharing
  area (the same patch counted twice).

In [ ]:
pos = raw[raw["classifica"] == 1]
parts = raw.geometry.apply(lambda g: len(g.geoms) if g.geom_type == "MultiPolygon" else 1)
dup = raw.geometry.normalize().to_wkb().duplicated(keep=False)
pairs = gpd.sjoin(pos[["geometry"]], pos[["geometry"]], predicate="overlaps")
pd.DataFrame({
    "invalid": raw.groupby("file").apply(lambda d: (~d.is_valid).sum()),
    "empty": raw.groupby("file").apply(lambda d: d.is_empty.sum()),
    "multipart_rows": parts.gt(1).groupby(raw["file"]).sum(),
    "pieces_in_multipart": parts.where(parts > 1, 0).groupby(raw["file"]).sum(),
    "exact_duplicates": dup.groupby(raw["file"]).sum(),
    "positives_overlapping_another": pd.Series(pairs.index.unique()).map(pos["file"]).value_counts(),
}).fillna(0).astype(int)

### How were the polygons made?

Hand-drawn polygons have vertices anywhere. Polygons made by turning
a raster (an image or a classified map) into vectors have vertices on
the raster's pixel grid. This checks, per file, the share of vertices
that fall on a regular grid of 0.25, 0.3, 0.5 and 1 m (measured from
the first vertex of the file).

In [ ]:
rows = {}
for f, d in raw.groupby("file"):
    xy = np.vstack([np.array(p.exterior.coords) for p in d.geometry.explode(index_parts=False)])
    rows[f] = {f"{step} m": round(np.mean((np.abs((xy - xy[0]) / step - np.round((xy - xy[0]) / step)) < 0.02).all(axis=1)), 3)
               for step in [0.25, 0.3, 0.5, 1.0]}
pd.DataFrame(rows).T.rename_axis("share of vertices on a grid of")

## 3. Sizes and shapes (positives only)

Area of each `classifica = 1` polygon, with multipart rows split into
pieces. A 10 m satellite pixel is 100 m²; polygons smaller than that
(`small`) are tiles of an image segmentation (~75 m² each, not
circles). **Circularity** = 4π·area /
perimeter² is 1 for a smooth circle and lower for irregular or
stair-stepped outlines.

In [ ]:
pieces = pos.explode(index_parts=False).reset_index(drop=True)
pieces["area_m2"] = pieces.area
pieces["circularity"] = 4 * np.pi * pieces.area / pieces.length ** 2
pieces["kind"] = np.where(pieces.area_m2 < 100, "small (<100 m²)", "patch")

print(pieces.groupby(["file", "kind"]).area_m2.describe(percentiles=[0.5]).round(1))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
bins = np.logspace(0, np.log10(pieces.area_m2.max()) + 0.1, 50)
COLOURS = {"patch": "tab:orange", "small (<100 m²)": "tab:blue"}
for k, d in pieces.groupby("kind"):
    axes[0].hist(d.area_m2, bins=bins, alpha=0.7, color=COLOURS[k], label=f"{k} (n={len(d)})")
axes[0].axvline(100, color="k", ls=":", lw=1); axes[0].text(110, axes[0].get_ylim()[1] * 0.9, "1 pixel", fontsize=8)
axes[0].set(xscale="log", xlabel="area (m², log)", ylabel="polygons", title="Area of positive polygons"); axes[0].legend()
axes[1].hist(pieces.circularity, bins=50, color="tab:gray")
axes[1].set(xlabel="circularity (1 = circle)", ylabel="polygons", title="Shape")
plt.tight_layout()

## 4. Where they are

Centroids of positive polygons over continental Portugal (left) and
the Coimbra district (right), coloured by kind. For the small
polygons in the district: how many 1 km cells they fall in, and the
distance from each to the nearest other small polygon (if they are
spread out, it is large; if they cover patches densely, it is close to
their 10 m diameter).

In [ ]:
pieces["in_district"] = pieces.intersects(district)
print(pieces.groupby(["in_district", "kind"]).size().unstack(fill_value=0))
print("area in the district (ha):", round(pieces.loc[pieces.in_district, "area_m2"].sum() / 1e4, 1))
small = gpd.GeoDataFrame(geometry=pieces[pieces.in_district & (pieces.kind != "patch")].centroid.values, crs=3763)
print("small polygons in the district fall in", len(set(zip(small.geometry.x // 1000, small.geometry.y // 1000))), "cells of 1 km")
nn = gpd.sjoin_nearest(small, small, distance_col="d", exclusive=True)["d"]
print("distance to the nearest other small polygon (m):", nn.quantile([0.1, 0.5, 0.9]).round(1).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
for ax, d in zip(axes, [pieces, pieces[pieces.in_district]]):
    gpd.GeoSeries([district], crs=3763).boundary.plot(ax=ax, color="black", lw=0.8)
    for k, c in COLOURS.items():
        d[d.kind == k].centroid.plot(ax=ax, markersize=3, color=c, label=k)
    ax.set_axis_off(); ax.legend(markerscale=4)
axes[0].set_title("All positives"); axes[1].set_title("Coimbra district")
plt.tight_layout()

### Overlap with iNaturalist

**What:** how many *A. dealbata* presences from iNaturalist (cleaned file,
from `02_citizen_science/acacia_observations.ipynb`) lie inside or
near (50 m, 100 m) a positive polygon in the district, and how many
of those are Observer A's; and in how many 10 km cells the positives
fall.
**Why:** the supervisor confirmed iNaturalist did not influence the
collection, so the labels are independent. But if experts and citizen
scientists visited the same places, testing the citizen-science model
on these polygons would evaluate it where it was trained. Few 10 km
cells also means few independent blocks for a spatial train/test
split.

In [ ]:
acacia = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
acacia = gpd.GeoDataFrame(acacia, geometry=gpd.points_from_xy(acacia.longitude, acacia.latitude), crs=4326).to_crs(3763)
acacia = acacia[acacia.within(district)]
pos_d = pos[pos.intersects(district)]
c = pos_d.geometry.centroid
print("positives in the district fall in", len(set(zip(c.x // 10_000, c.y // 10_000))), "cells of 10 km")
print(f"iNaturalist presences in the district: {len(acacia)}")
rows = []
for buf in (0, 50, 100):
    near = gpd.sjoin(acacia[["id", "observer", "geometry"]], gpd.GeoDataFrame(geometry=pos_d.buffer(buf), crs=3763),
                     predicate="within").drop_duplicates("id")
    rows.append(dict(distance=f"within {buf} m" if buf else "inside", presences=len(near),
                     pct_observer_A=round(100 * (near.observer == "Observer A").mean(), 1)))
pd.DataFrame(rows)

**Result.** The positives fall in only 15 cells of 10 km. 22
cleaned iNaturalist *A. dealbata* presences lie inside a polygon (all
Observer A's), 162 within 50 m (92.6% Observer A's) and 295 within
100 m. The experts and Observer A visited the same places: training
and test must be split by spatial blocks.

## 5. What COS 2023 says is under each polygon

The supervisor warned that many `classifica = 0` polygons lie on water
or buildings. If the negatives are mostly water and buildings, a model
learns "vegetation vs. not vegetation" instead of "*A. dealbata* vs.
other plants". This checks it: for each polygon, the COS 2023 class at
a point inside it (`representative_point`), grouped as below.

COS has a minimum mapping unit of 1 ha, so for a 75 m² segment it says
what dominates the surrounding hectare, not the segment itself.

In [ ]:
COS = ROOT / "data" / "interim" / "spatial" / "COS2023v1-S2.gpkg"
pts = gpd.GeoDataFrame(raw[["file", "classifica"]], geometry=raw.geometry.make_valid().representative_point(), crs=3763)
pts["small"] = raw.area < 100
blocks = pd.Series(list(zip(pts.geometry.x // 10_000, pts.geometry.y // 10_000)), index=pts.index)
joined = []
for (bx, by), d in pts.groupby(blocks):  # read COS only around the polygons, 10 km at a time
    cos = gpd.read_file(COS, layer="COS2023v1", bbox=(bx * 1e4, by * 1e4, (bx + 1) * 1e4, (by + 1) * 1e4),
                        columns=["COS23_n4_C", "COS23_n4_L"])
    joined.append(gpd.sjoin(d, cos, predicate="within", how="left").drop(columns="index_right"))
pts = pd.concat(joined)
pts = pts[~pts.index.duplicated()]

def cos_group(code):
    if code.startswith("1"): return "1 artificial (buildings, industry, roads)"
    if code[0] in "234": return "2 agriculture and pasture"
    if code == "5.1.1.6": return "3 eucalyptus forest"
    if code == "5.1.1.7": return "4 acacia forest (COS)"
    if code.startswith("5.1.2"): return "5 pine forest"
    if code.startswith("5"): return "6 other forest"
    if code.startswith("6"): return "7 shrubland (matos)"
    return "8 water, wetlands, bare"
pts["cos_group"] = pts["COS23_n4_C"].map(cos_group)

neg, pos_ = pts[pts.classifica == 0], pts[pts.classifica == 1]
table = pd.DataFrame({
    "negatives": neg.cos_group.value_counts(),
    "negatives %": neg.cos_group.value_counts(normalize=True).mul(100).round(1),
    "of which small": neg[neg.small].cos_group.value_counts(),
    "positives": pos_.cos_group.value_counts(),
    "positives %": pos_.cos_group.value_counts(normalize=True).mul(100).round(1),
}).fillna(0).sort_index()
table.loc["total"] = table.sum().where(~table.columns.str.endswith("%"), 100)
table.astype({c: int for c in table.columns if not c.endswith("%")})

The most frequent COS classes under the negatives, to see what is
inside each group:

In [ ]:
neg.COS23_n4_L.value_counts().head(12)

# Part 2 — Cleaning (preprocessing)

Sections 1–5 only looked at the data. From here the data are changed,
one step at a time, and saved to
`data/interim/expert_reference/expert_polygons_clean.gpkg`. The raw
files are never modified. Each step says what it does, why, and what
changed.

What is **not** done here: turning the polygons into 10 m pixels.
That needs the satellite grid, so it belongs to
`05_data_integration/`.

## 6. Keep only `classifica = 1` rows

**What:** drop rows with an empty `classifica`, and drop
`classifica = 0` rows entirely.
**Why:** without a label they cannot be used at all. The
`classifica = 0` polygons are dropped per the supervisor's decision
(2026-11-04): they are not reliable absences (44% sit in artificial
areas, section 5), so using them taught "vegetation vs. not
vegetation" instead of "*A. dealbata* vs. other plants" — see
[`thesis/notes.md`](../../thesis/notes.md#usar-os-dados-de-especialistas-com-limpeza).
From here on, every row is a presence.

In [ ]:
clean = raw.copy()
before = len(clean)
clean = clean[clean["classifica"] == 1].copy()
print(f"{before} → {len(clean)} rows (dropped {before - len(clean)}: {raw['classifica'].isna().sum()} unlabelled, {(raw['classifica'] == 0).sum()} classifica = 0)")

**Result.** 5 024 rows dropped (11 unlabelled, 5 013 `classifica = 0`);
2 470 `classifica = 1` polygons remain (1 849 in the Coimbra district,
621 elsewhere).

## 7. Fix invalid geometries

**What:** `make_valid` repairs the polygons whose outline touches
itself (section 2 found 531 invalid rows over all of `raw`; only 49
of those are `classifica = 1` and matter now that section 6 dropped
the rest). Where two pixels of the original raster touched only at a
corner, the "figure 8" is split into two pieces, so the result can be
a `MultiPolygon` or a `GeometryCollection` that also holds stray
lines or points of zero area. Only the polygon parts are kept.
**Why:** GIS operations (area, intersection, "is this point inside?")
can fail or give wrong answers on invalid geometries.
**Check:** the total area must not change.

In [ ]:
from shapely.geometry import MultiPolygon

def polygons_only(g):
    if g.geom_type in ("Polygon", "MultiPolygon"):
        return g
    return MultiPolygon([p for part in g.geoms for p in getattr(part, "geoms", [part]) if p.geom_type == "Polygon"])

area_before = clean.area.sum()
n_invalid = (~clean.is_valid).sum()
clean["geometry"] = clean.make_valid().apply(polygons_only)
print(f"invalid before: {n_invalid}, after: {(~clean.is_valid).sum()}")
print(f"total area before: {area_before:,.1f} m², after: {clean.area.sum():,.1f} m²")
print("geometry types:", clean.geom_type.value_counts().to_dict())

**Result.** No invalid geometries remain among the 2 470 positives and
the total area is unchanged (49 were invalid before the repair). The
repair only changed how the outlines are written, not the labelled
areas. Every row is now a `Polygon` or `MultiPolygon`.

## 8. Overlaps between polygons

**What:** look for `classifica = 1` polygons that share area with
another one (that area would be counted twice). Only positive ×
positive matters now; the positive × negative and negative ×
negative checks from before section 6 dropped the negatives are no
longer relevant.
**Why:** section 2 already found 2 positives overlapping over the
whole dataset; checking again on the cleaned geometries confirms
whether the `make_valid` repair changed that.

In [ ]:
def overlaps(a, b, same):
    x = gpd.overlay(a[["geometry"]].reset_index(), b[["geometry"]].reset_index(), how="intersection", keep_geom_type=True)
    if same:
        x = x[x["index_1"] < x["index_2"]]
    return len(x), round(x.area.sum(), 2)

n, area = overlaps(clean, clean, True)
pd.DataFrame([[n, area]], index=["positive × positive"], columns=["pairs", "shared area (m²)"])

**Result.** Negligible: 1 pair of positives shares 1.6 m² — the same
pair found in section 2, unaffected by the `make_valid` repair.
Nothing is changed: it does not affect a 10 m pixel's label.

## 9. Add the attributes needed later

The raw files have only `classifica`, which is now constant
(`classifica = 1` for every remaining row, so it is dropped). Each
polygon gets:
- `source`: the file it came from (`TNE` or `notTNE`);
- `area_m2`;
- `segment`: `True` if smaller than one 10 m pixel (< 100 m²), i.e. a
  tile of the image segmentation;
- `in_district`: intersects the Coimbra district;
- `cos_code`, `cos_label`, `cos_group`: the COS 2023 class at a point
  inside it (section 5) — informational only now (e.g. 54% fall in
  COS "Florestas de acácias"), not used to assign a label.

In [ ]:
clean["source"] = clean["file"]
clean["area_m2"] = clean.area.round(1)
clean["segment"] = clean["area_m2"] < 100
clean["in_district"] = clean.intersects(district)
clean = clean.join(pts[["COS23_n4_C", "COS23_n4_L", "cos_group"]].rename(
    columns={"COS23_n4_C": "cos_code", "COS23_n4_L": "cos_label"}))
clean = clean[["source", "area_m2", "segment", "in_district", "cos_code", "cos_label", "cos_group", "geometry"]]
clean.head()

## 10. Save

Saved as one layer, in EPSG:3763. Every row is now a `classifica = 1`
presence (see section 6); it can be opened in QGIS directly.

In [ ]:
OUT = ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg"
OUT.parent.mkdir(parents=True, exist_ok=True)
clean.reset_index(drop=True).to_file(OUT, layer="expert_polygons_clean", driver="GPKG")
print(OUT.relative_to(ROOT), len(clean), "polygons")

## Summary

**1. Contents.** 7 494 rows: 2 470 `classifica = 1` (*A. dealbata*),
5 013 `classifica = 0`, 11 empty (all empty ones in `notTNE`). Only
`classifica` and the geometry: no date, method, author or species.

**2. Geometry quality and origin.**
- 531 invalid geometries (530 in TNE, 48 of them positives), all
  self-intersections. No empty, multipart or duplicate rows; 2
  positives overlap another.
- **TNE was made from a raster:** 94.6% of its vertices sit on a
  0.3 m grid; `notTNE` vertices sit on no grid (drawn as vectors).
  The self-intersections are typical of raster-to-vector conversion
  (pixels touching at a corner). The raster is OrtoSat 2023 (30 cm),
  per the supervisor.

**3. Sizes.** 1 364 positives are smaller than one 10 m pixel (1 353
in TNE, median 74.5 m², segmentation tiles); the other
1 106 are patches (median ~2 000 m², up to 26 ha).

**4. Where.** 1 849 positives in the Coimbra district (244 ha), 621
outside. In the district, all 1 353 small polygons fall in only 14
cells of 1 km, and half of them are within 8.9 m of another one:
they cover a few areas densely, almost touching, rather than being
spread-out field points. So they count as 1 353 polygons but carry
much less independent information. The positives fall in only 15
cells of 10 km. 162 cleaned iNaturalist *A. dealbata* presences lie
within 50 m of a positive (92.6% Observer A's; all 22 inside one are): same places,
independent labels.

**5. COS under the polygons.** Of the 5 013 negatives, 43.6% are in
artificial areas (buildings, industry) and almost none in water (3).
The other 56% are in vegetation: pine 941, other forest 496, acacia
forest (COS) 349, eucalyptus 308, agriculture 643, shrubland 86. 54%
of the positives fall in COS "Florestas de acácias", the rest mostly
in pine, eucalyptus and other forest (COS maps the dominant cover per
hectare). This 43.6%-in-artificial-areas figure is what the
supervisor later (2026-11-04) pointed to when dropping the negatives
entirely (part 2, section 6) — using them risked teaching "vegetation
vs. not vegetation" rather than "*A. dealbata* vs. other plants".

**Part 2 — cleaning.** Section 6 now drops every `classifica = 0` and
unlabelled row (decision, supervisor, 2026-11-04 — see
[`thesis/notes.md`](../../thesis/notes.md#usar-os-dados-de-especialistas-com-limpeza)),
leaving only the 2 470 `classifica = 1` polygons; 49 invalid
geometries among them (of the 531 found in section 2 over the whole
raw dataset) are repaired with `make_valid` (area unchanged); the
remaining positive × positive overlap is negligible (1 pair, 1.6 m²);
attributes added (`source`, `area_m2`, `segment`, `in_district`, COS
class — informational only, no `use`/`label` column, since every row
is now a presence). Result: 2 470 polygons (1 849 in the district, 621
outside) in `data/interim/expert_reference/expert_polygons_clean.gpkg`.
Next, in `05_data_integration/`: turn the polygons into 10 m pixels.

**Answered by the supervisor (2026-10-28):** 2023, field visits and
manual visual interpretation of OrtoSat 2023 (30 cm, which matches the
0.3 m grid). The small polygons are probably segments of an unsupervised
segmentation, each labelled *A. dealbata* or not: not field points.
In the same 14 cells there are 4 256 small `classifica = 0` segments
(dropped, see above).